# Asymmetry-Induced Judge Drift — Experiment Analysis Notebook

**Research question:** Does persistent structural asymmetry in evaluation order induce measurable drift, inflation, or reciprocity bias in LLM judging behaviour?

**Protocol:** Model A is always evaluated *first* by B, C, D — then A evaluates B, C, D. This structural constant is the experimental manipulation.

**This notebook covers:**
1. Environment setup & GPU verification
2. Global configuration & reproducibility controls
3. Model registry & version hashing
4. Task template definitions & prompt preview
5. Scoring rubric & judge prompt engineering
6. JSONL logging schema & writer
7. Checkpoint manager
8. GPU memory manager & sequential model loader
9. Response generation engine
10. Response cache layer
11. Judge execution engine
12. Asymmetry protocol orchestrator
13. Bias metric computations
14. Visualization suite (8 publication-grade figures)
15. Dry-run mode & consistency assertions
16. Main experiment loop with resume support

## Section 1: Environment Setup & Dependency Installation

In [ ]:
# ── Install / verify dependencies ──────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Uncomment on first run in a fresh environment:
# for pkg in [
#     "torch", "transformers>=4.40.0", "bitsandbytes>=0.43.1",
#     "accelerate>=0.29.3", "huggingface-hub", "datasets",
#     "matplotlib", "seaborn", "pandas", "numpy", "scipy",
#     "pyyaml", "tqdm", "jsonlines", "colorlog", "psutil",
# ]:
#     install(pkg)

import os, sys, gc, json, time, random, hashlib, re, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

# ── Working directory ───────────────────────────────────────────────────────
SCRIPT_DIR = Path(".").resolve()
# If running from repo root, point to the sub-directory
if (SCRIPT_DIR / "universe_B_v2").exists():
    BASE_DIR = SCRIPT_DIR / "universe_B_v2"
else:
    BASE_DIR = SCRIPT_DIR

for sub in ["checkpoints", "logs", "cache", "figures", "results", "configs", "src", "tasks"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE_DIR))
print(f"BASE_DIR: {BASE_DIR}")

# ── GPU check ────────────────────────────────────────────────────────────────
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    if CUDA_AVAILABLE:
        dev = torch.cuda.get_device_properties(0)
        VRAM_TOTAL_GB = dev.total_memory / 1e9
        print(f"✓ CUDA available  |  Device: {dev.name}  |  VRAM: {VRAM_TOTAL_GB:.1f} GB")
        VRAM_LIMIT_GB = 14.0  # Safety threshold for a T4 (16 GB)
    else:
        print("⚠  CUDA not available — will use CPU (slow, for analysis runs only)")
        VRAM_TOTAL_GB = 0.0
        VRAM_LIMIT_GB = 0.0
except ImportError:
    CUDA_AVAILABLE = False
    VRAM_TOTAL_GB = 0.0
    print("⚠  PyTorch not installed.")

## Section 2: Global Configuration & Reproducibility Controls

In [ ]:
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple

@dataclass(frozen=True)
class ExperimentConfig:
    # ── Reproducibility ──────────────────────────────
    GLOBAL_SEED: int = 42
    TEMPERATURE: float = 0.05        # Near-deterministic for judging
    GEN_TEMPERATURE: float = 0.7     # For response generation
    MAX_NEW_TOKENS: int = 512
    TOP_P: float = 0.9
    REPETITION_PENALTY: float = 1.1

    # ── Task scale ───────────────────────────────────
    NUM_TEMPLATES: int = 10
    VARIATIONS_PER_TEMPLATE: int = 100
    TOTAL_ITERATIONS: int = 1000     # 10 × 100

    # ── Runtime ──────────────────────────────────────
    CHECKPOINT_INTERVAL: int = 10
    DRY_RUN: bool = False
    DRY_RUN_ITERATIONS: int = 5
    PARSE_RETRIES: int = 3

    # ── Asymmetry ────────────────────────────────────
    ASYMMETRY_FLAG: bool = True      # Model A always judged first
    MODEL_JUDGED_FIRST: str = "model_a"

    # ── Quantization ─────────────────────────────────
    QUANTIZATION_METHOD: str = "nf4_bnb_double_quant"

    # ── Score range ──────────────────────────────────
    SCORE_MIN: int = 0
    SCORE_MAX: int = 100


CFG = ExperimentConfig()


def set_global_seed(seed: int) -> None:
    """Set all random sources to a deterministic state."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    except ImportError:
        pass
    print(f"Global seed set: {seed}")


set_global_seed(CFG.GLOBAL_SEED)

# Save config snapshot
cfg_snapshot = {
    **asdict(CFG),
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "cuda_available": CUDA_AVAILABLE,
    "device_name": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "cpu",
}
cfg_path = BASE_DIR / "configs" / "experiment_config.json"
with open(cfg_path, "w") as f:
    json.dump(cfg_snapshot, f, indent=2)
print(f"Config snapshot saved → {cfg_path}")
print(f"Total iterations: {CFG.TOTAL_ITERATIONS}  |  Templates: {CFG.NUM_TEMPLATES}  |  Variations/template: {CFG.VARIATIONS_PER_TEMPLATE}")

## Section 3: Model Registry & Version Hashing

In [ ]:
MODEL_REGISTRY: Dict[str, Dict] = {
    "model_a": {
        "hf_name": "microsoft/Phi-3-mini-4k-instruct",
        "display_name": "Model_A",
        "role": "primary_subject",      # judged first; also judges
        "chat_template": "phi3",
        "version_hash": None,           # filled by get_model_version_hash()
    },
    "model_b": {
        "hf_name": "stabilityai/stablelm-3b-4e1t",
        "display_name": "Model_B",
        "role": "judge_and_peer",
        "chat_template": "stablelm",
        "version_hash": None,
    },
    "model_c": {
        "hf_name": "google/gemma-2b-it",
        "display_name": "Model_C",
        "role": "judge_and_peer",
        "chat_template": "gemma",
        "version_hash": None,
    },
    "model_d": {
        "hf_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "display_name": "Model_D",
        "role": "judge_and_peer",
        "chat_template": "tinyllama",
        "version_hash": None,
    },
}

_model_hash_cache_path = BASE_DIR / "configs" / "model_hashes.json"


def get_model_version_hash(model_key: str) -> str:
    """
    Fetch the latest commit SHA from HuggingFace Hub API.
    Falls back to a deterministic hash of the model name if offline.
    Caches results to configs/model_hashes.json.
    """
    # Load existing cache
    if _model_hash_cache_path.exists():
        with open(_model_hash_cache_path) as f:
            cache = json.load(f)
    else:
        cache = {}

    if model_key in cache:
        return cache[model_key]

    hf_name = MODEL_REGISTRY[model_key]["hf_name"]
    version_hash = None

    try:
        from huggingface_hub import model_info
        info = model_info(hf_name)
        sha = getattr(info, "sha", None) or getattr(info, "id", hf_name)
        version_hash = hashlib.sha256(str(sha).encode()).hexdigest()[:16]
        print(f"  {model_key}: fetched hash from HF Hub → {version_hash}")
    except Exception as e:
        # Offline fallback: hash the name string
        version_hash = hashlib.sha256(hf_name.encode()).hexdigest()[:16]
        print(f"  {model_key}: offline fallback hash → {version_hash}  ({e})")

    cache[model_key] = version_hash
    with open(_model_hash_cache_path, "w") as f:
        json.dump(cache, f, indent=2)

    return version_hash


# Populate version hashes
print("Resolving model version hashes...")
for key in MODEL_REGISTRY:
    MODEL_REGISTRY[key]["version_hash"] = get_model_version_hash(key)

print("\nModel Registry:")
for key, info in MODEL_REGISTRY.items():
    print(f"  {key:10s}  {info['hf_name']:45s}  hash={info['version_hash']}")

# ── BitsAndBytes 4-bit config (defined here for documentation) ────────────
try:
    import transformers
    from transformers import BitsAndBytesConfig
    BNB_CONFIG = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16 if CUDA_AVAILABLE else None,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=False,
    )
    print(f"\nBitsAndBytes 4-bit config ready (method={CFG.QUANTIZATION_METHOD})")
except Exception as e:
    BNB_CONFIG = None
    print(f"BitsAndBytes not available: {e}")

## Section 4 — Task Template Definitions

Generate all **1000 task prompt variations** (10 template families × 100 seeded variations).  
Each variation is deterministically reproducible via `variation_seed = GLOBAL_SEED * 1_000_000 + template_id * 1_000 + variation_id`.

In [ ]:
import sys, json
from pathlib import Path

# Add repository root to path
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tasks.templates import TaskRegistry, TaskVariation

GLOBAL_SEED = 42
NUM_TEMPLATES = 10
NUM_VARIATIONS = 100

# Instantiate registry — all seeds derived deterministically
registry = TaskRegistry(global_seed=GLOBAL_SEED)

# Materialize all 1000 variations
all_variations: list[TaskVariation] = list(registry.iter_all(
    num_templates=NUM_TEMPLATES,
    num_variations=NUM_VARIATIONS
))

print(f"Total variations generated: {len(all_variations)}")
print(f"Expected: {NUM_TEMPLATES * NUM_VARIATIONS}")

# Save to JSONL for caching
task_cache_path = REPO_ROOT / "logs" / "task_variations.jsonl"
task_cache_path.parent.mkdir(exist_ok=True)
with open(task_cache_path, "w") as fh:
    for v in all_variations:
        fh.write(json.dumps({
            "iteration_id": v.iteration_id,
            "template_id": v.template_id,
            "variation_id": v.variation_id,
            "family_name": v.family_name,
            "prompt": v.prompt,
            "seed": v.seed,
        }) + "\n")
print(f"Saved → {task_cache_path}")

# Preview 2 examples per family
print("\n--- Example prompts (1 per family) ---")
seen_families: set[str] = set()
for v in all_variations:
    if v.family_name not in seen_families:
        seen_families.add(v.family_name)
        print(f"\n[{v.family_name}] iter={v.iteration_id}")
        print(v.prompt[:300], "..." if len(v.prompt) > 300 else "")

## Section 5 — Scoring Rubric & Judge Prompt Engineering

The rubric uses **5 weighted dimensions** totalling 100 points.  
Judge prompts include two calibrated few-shot examples before the live response to anchor scoring and reduce positional bias.

In [ ]:
from tasks.rubrics import (
    build_judge_prompt,
    parse_judge_output,
    _DIMENSION_ANCHORS,
    _FEW_SHOT_EXAMPLES,
)

# ── Rubric dimensions with weights ──────────────────────────────────────
RUBRIC_DIMENSIONS = {
    "clarity_coherence":       0.20,
    "depth_of_reasoning":      0.25,
    "practical_applicability": 0.20,
    "internal_consistency":    0.20,
    "persuasiveness":          0.15,
}
assert abs(sum(RUBRIC_DIMENSIONS.values()) - 1.0) < 1e-9, "Weights must sum to 1"

print("Rubric dimensions and weights:")
for dim, weight in RUBRIC_DIMENSIONS.items():
    print(f"  {dim:<30} {weight*100:5.1f}%")
print()
print(f"Anchors defined for {len(_DIMENSION_ANCHORS)} dimensions")
print(f"Few-shot examples: {len(_FEW_SHOT_EXAMPLES)}")

# ── Preview: build one full judge prompt ────────────────────────────────
sample_task   = all_variations[0].prompt
sample_answer = (
    "This is a sample model response for demonstration purposes. "
    "The policy in question requires careful consideration of multiple factors "
    "including economic impact, social equity, and long-term sustainability. "
    "A balanced approach would weigh short-term costs against long-term gains."
)

judge_prompt = build_judge_prompt(
    task_prompt=sample_task,
    response_text=sample_answer,
    rubric_dimensions=RUBRIC_DIMENSIONS,
)

print(f"\nJudge prompt length: {len(judge_prompt)} chars")
print("=" * 70)
print(judge_prompt[:1200], "\n[... truncated ...]")

## Section 6 — JSONL Logging Schema & Writer

Three independent JSONL streams provide an immutable audit trail:

| Stream | File | Key fields |
|---|---|---|
| Responses | `responses_cache.jsonl` | `iteration_id`, `model_id`, `response_text`, `generation_seed` |
| Scores | `scores_log.jsonl` | `iteration_id`, `judge_id`, `target_id`, `weighted_total`, `dimensions` |
| Events | `events_log.jsonl` | `event_type`, `timestamp`, `metadata` |

In [ ]:
import time, json, tempfile, os
from typing import TypedDict, Any
from pathlib import Path
from src.logging_utils import ExperimentLogger, load_jsonl, setup_python_logging

# ── Schema definitions (TypedDicts) ────────────────────────────────────
class ResponseRecord(TypedDict):
    iteration_id:    int
    model_id:        str
    response_text:   str
    generation_seed: int
    generation_time_sec: float
    timestamp:       str

class ScoreRecord(TypedDict):
    iteration_id:    int
    judge_id:        str
    target_id:       str
    weighted_total:  float
    dimensions:      dict[str, float]
    parse_success:   bool
    asymmetry_position: int   # 0 = A judged by others; 1,2 = A judges others
    timestamp:       str

class EventRecord(TypedDict):
    event_type: str
    timestamp:  str
    metadata:   dict[str, Any]

# ── Demonstrate write + read roundtrip ─────────────────────────────────
demo_log_dir = REPO_ROOT / "logs"
logger = ExperimentLogger(log_dir=demo_log_dir, run_id="demo")
setup_python_logging(level="WARNING")

# Write a synthetic response record
logger.log_response(
    iteration_id=0,
    model_id="A",
    response_text="Demo response text.",
    generation_seed=12345,
    generation_time_sec=0.5,
)

# Write a synthetic score record
logger.log_score(
    iteration_id=0,
    judge_id="B",
    target_id="A",
    weighted_total=72.5,
    dimensions={"clarity_coherence": 75, "depth_of_reasoning": 70,
                "practical_applicability": 72, "internal_consistency": 74,
                "persuasiveness": 68},
    parse_success=True,
    asymmetry_position=0,
)

# Write an event record
logger.log_event("demo_event", {"note": "Section 6 validation"})

# Read back
resp_records = load_jsonl(demo_log_dir / "demo_responses_cache.jsonl")
score_records = load_jsonl(demo_log_dir / "demo_scores_log.jsonl")
event_records = load_jsonl(demo_log_dir / "demo_events_log.jsonl")

print(f"Response records written: {len(resp_records)}")
print(f"Score records written:    {len(score_records)}")
print(f"Event records written:    {len(event_records)}")
print("\nSample score record:")
import pprint; pprint.pprint(score_records[0])

## Section 7 — Checkpoint Manager

Atomic checkpoint persistence (`mkstemp` + `os.replace`) guarantees the checkpoint file is never left in a partially written state, enabling safe resume after any failure point.

In [ ]:
import random, numpy as np
from src.checkpointing import CheckpointManager, Checkpoint

ckpt_dir = REPO_ROOT / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

# Create a fresh checkpoint manager
cm = CheckpointManager(checkpoint_dir=ckpt_dir, total_iterations=1000)
ckpt = cm.load_or_create()

print(f"Starting from iteration: {ckpt.next_iteration}")
print(f"Progress: {ckpt.progress_fraction:.1%}")
print(f"Complete: {ckpt.is_complete}")

# Simulate completing iteration 0
random.seed(99); np.random.seed(99)
ckpt = cm.update(ckpt, completed_iteration=0, duration_sec=3.14)
cm.save(ckpt)
print(f"\nAfter saving iter 0 → next_iteration = {ckpt.next_iteration}")

# Reload from disk — demonstrates persistence
ckpt2 = cm.load_or_create()
print(f"Reloaded checkpoint → next_iteration = {ckpt2.next_iteration}")
print(f"Stored duration for iter 0: {ckpt2.iteration_durations_sec.get(0)} sec")

# Demonstrate RNG state serialization
rng_state_saved = {
    "python": list(random.getstate()[1]),
    "numpy":  [int(x) for x in np.random.get_state()[1]],
}
print(f"\nPython RNG state vector length: {len(rng_state_saved['python'])}")
print(f"NumPy RNG state vector length:  {len(rng_state_saved['numpy'])}")

## Section 8 — GPU Memory Manager & Sequential Model Loader

Only **one model** resides in VRAM at any time.  
The `ModelHandle` context manager calls `aggressive_gpu_clear()` (empty cache + gc.collect) on `__exit__`, verified by `log_gpu_memory()` before and after.

In [ ]:
import gc
import torch
from src.models import (
    ModelHandle,
    aggressive_gpu_clear,
    log_gpu_memory,
    build_bnb_config,
    compute_model_version_hash,
    create_handles,
)

# ── GPU availability check ───────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    log_gpu_memory("baseline")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc_gb  = torch.cuda.memory_allocated(0) / 1e9
    print(f"Total VRAM: {total_gb:.2f} GB")
    print(f"Allocated:  {alloc_gb:.3f} GB")
else:
    print("Running CPU-only — GPU checks skipped.")

# ── Model handle overview (no actual load in dry run) ───────────────────
print("\nModel handles that would be created:")
for mid, mdata in MODEL_REGISTRY.items():
    print(f"  Model {mid}: {mdata['hf_name']} [{mdata['chat_template']}]")

# ── BitsAndBytes config ──────────────────────────────────────────────────
if device == "cuda":
    try:
        bnb = build_bnb_config(BNB_CONFIG)
        print(f"\nBitsAndBytes config: load_in_4bit=True, quant_type={BNB_CONFIG['bnb_4bit_quant_type']}")
    except Exception as e:
        print(f"\nbitsandbytes unavailable in this env: {e}")

# ── Sequential loading contract ──────────────────────────────────────────
print("""
Sequential loading contract:
  1. Load model X
  2. Run generate() calls
  3. Unload model X  ← model.cpu(), del model/tokenizer, aggressive_gpu_clear()
  4. Load model X+1
  At most 1 model in VRAM at any time.
""")

## Section 9 — Response Generation Engine

`format_prompt()` converts `(system, user, template_name)` → a chat-formatted string.  
Each call to `ModelHandle.generate()` accepts an explicit `seed` so generation is byte-reproducible given the same checkpoint.

In [ ]:
from src.models import format_prompt

# ── Demonstrate chat template formatting ────────────────────────────────
SYSTEM_MSG = (
    "You are a thoughtful analyst. Respond clearly and concisely, "
    "with structured reasoning."
)
USER_MSG = all_variations[0].prompt  # first task variation

print("Chat template formatting (first 400 chars per template):\n")
for template_name in ["phi3", "stablelm", "gemma", "tinyllama"]:
    formatted = format_prompt(SYSTEM_MSG, USER_MSG, template_name)
    print(f"[{template_name}]")
    print(formatted[:400])
    print("...\n" if len(formatted) > 400 else "\n")

# ── Seed derivation scheme ────────────────────────────────────────────────
def derive_iteration_seed(global_seed: int, iteration_id: int) -> int:
    """Knuth multiplicative hash for per-iteration seeds."""
    return (global_seed ^ (iteration_id * 2_654_435_761)) & 0xFFFF_FFFF

def derive_model_seed(iteration_seed: int, model_idx: int) -> int:
    return (iteration_seed ^ (model_idx * 1_000_003)) & 0xFFFF_FFFF

print("Sample seed derivations:")
for it in [0, 1, 499, 999]:
    it_seed = derive_iteration_seed(GLOBAL_SEED, it)
    print(f"  iter {it:>4}: iter_seed={it_seed:>12}  "
          f"model_A_seed={derive_model_seed(it_seed, 0):>12}")

## Section 10 — Response Cache Layer

Responses are keyed by `sha256(model_id + "|" + iteration_id + "|" + generation_seed)`.  
On resume, cached responses skip the model-load entirely, dramatically reducing wall-clock time for partial re-runs.

In [ ]:
import hashlib
from src.logging_utils import load_jsonl

class ResponseCache:
    """Thin in-memory cache backed by responses_cache.jsonl."""

    def __init__(self, jsonl_path: Path):
        self._path = jsonl_path
        self._store: dict[str, str] = {}
        self._hits = 0
        self._misses = 0
        self._load()

    def _load(self) -> None:
        if not self._path.exists():
            return
        for rec in load_jsonl(self._path):
            key = self._make_key(rec["model_id"], rec["iteration_id"], rec["generation_seed"])
            self._store[key] = rec["response_text"]

    @staticmethod
    def _make_key(model_id: str, iteration_id: int, seed: int) -> str:
        raw = f"{model_id}|{iteration_id}|{seed}"
        return hashlib.sha256(raw.encode()).hexdigest()

    def get(self, model_id: str, iteration_id: int, seed: int) -> str | None:
        key = self._make_key(model_id, iteration_id, seed)
        val = self._store.get(key)
        if val is not None:
            self._hits += 1
        else:
            self._misses += 1
        return val

    def put(self, model_id: str, iteration_id: int, seed: int, text: str) -> None:
        key = self._make_key(model_id, iteration_id, seed)
        self._store[key] = text

    @property
    def stats(self) -> dict:
        total = self._hits + self._misses
        return {
            "size":      len(self._store),
            "hits":      self._hits,
            "misses":    self._misses,
            "hit_rate":  self._hits / total if total else 0.0,
        }


# ── Demo ──────────────────────────────────────────────────────────────────
cache_path = REPO_ROOT / "logs" / "responses_cache.jsonl"
rc = ResponseCache(cache_path)

# Simulate miss then put then hit
val = rc.get("A", 0, 12345)
print(f"Initial lookup (miss expected): {val}")
rc.put("A", 0, 12345, "Cached response text.")
val2 = rc.get("A", 0, 12345)
print(f"After put (hit expected):       {val2}")
print(f"Cache stats: {rc.stats}")

## Section 11 — Judge Execution Engine

`parse_judge_output()` extracts dimension scores from the model's raw text via JSON deserialization with regex fallback.  
If parsing fails after `parse_retries` attempts, scores default to **50.0** (neutral midpoint) and `parse_success=False` is flagged for post-hoc analysis.

In [ ]:
from tasks.rubrics import parse_judge_output, JudgeScore

# ── Simulate a well-formed judge output ─────────────────────────────────
well_formed_output = """\
Here is my evaluation:

```json
{
  "clarity_coherence": 78,
  "depth_of_reasoning": 65,
  "practical_applicability": 71,
  "internal_consistency": 80,
  "persuasiveness": 60
}
```

Overall the response is clear but lacks depth.
"""

score_good: JudgeScore = parse_judge_output(
    raw_output=well_formed_output,
    rubric_dimensions=RUBRIC_DIMENSIONS,
    attempt=0,
)
print("Well-formed parse:")
print(f"  parse_success    : {score_good.parse_success}")
print(f"  weighted_total   : {score_good.weighted_total:.2f}")
print(f"  dimensions       : {score_good.dimensions}")

# ── Simulate a malformed judge output (triggers fallback) ─────────────────
malformed_output = "I cannot evaluate this response adequately."
score_bad: JudgeScore = parse_judge_output(
    raw_output=malformed_output,
    rubric_dimensions=RUBRIC_DIMENSIONS,
    attempt=3,   # beyond retry limit → fallback
)
print("\nMalformed parse (fallback):")
print(f"  parse_success    : {score_bad.parse_success}")
print(f"  weighted_total   : {score_bad.weighted_total:.2f}  (fallback neutral)")

# ── Validate output contract ──────────────────────────────────────────────
assert 0.0 <= score_good.weighted_total <= 100.0
assert 0.0 <= score_bad.weighted_total  <= 100.0
assert set(score_good.dimensions.keys()) == set(RUBRIC_DIMENSIONS.keys())
print("\nAll assertions passed ✓")

## Section 12 — Asymmetry Protocol Orchestrator

The structural asymmetry is enforced by fixed evaluation ordering in `IterationProtocol.run_iteration()`:

| Phase | Step | Action |
|---|---|---|
| 1 | 1a | All 4 models generate responses independently |
| 2a | 2a-i | B judges A (`asymmetry_position=0`) |
| 2a | 2a-ii | C judges A (`asymmetry_position=1`) |
| 2a | 2a-iii | D judges A (`asymmetry_position=2`) |
| 2b | 2b-i | A judges B (`asymmetry_position=3`) |
| 2b | 2b-ii | A judges C (`asymmetry_position=4`) |
| 2b | 2b-iii | A judges D (`asymmetry_position=5`) |

Model A's response is always evaluated by others **before** A evaluates anyone else.

In [ ]:
from src.protocols import IterationProtocol

# ── Print the asymmetry ordering contract ──────────────────────────────
MODEL_IDS  = list(MODEL_REGISTRY.keys())   # ["A", "B", "C", "D"]
PEER_IDS   = [m for m in MODEL_IDS if m != "A"]

print("Phase 1 — Generation (all models, in parallel concept):")
for mid in MODEL_IDS:
    print(f"  Generate: Model {mid}")

print("\nPhase 2a — B, C, D each judge A (A is judged FIRST):")
for pos, judge in enumerate(PEER_IDS):
    print(f"  Judge {judge} → Target A  [asymmetry_position={pos}]")

print("\nPhase 2b — A judges B, C, D (after A has been seen):")
for pos, target in enumerate(PEER_IDS):
    print(f"  Judge A → Target {target}  [asymmetry_position={len(PEER_IDS)+pos}]")

total_scores_per_iter = len(PEER_IDS) * 2   # each peer pair judged once each way
print(f"\nScores per iteration: {total_scores_per_iter}")
print(f"Total scores across 1000 iterations: {total_scores_per_iter * 1000}")

# ── Asymmetry flags ───────────────────────────────────────────────────────
print("\nAsymmetry flags:")
print("  is_subject_of_judgment (A being judged): positions 0,1,2")
print("  is_judge (A judging others):             positions 3,4,5")
print("  Structural bias window: A processed BEFORE exposure to others' scores")

## Section 13 — Bias Metric Computations

Seven metrics measure distinct facets of asymmetry-induced drift:

| ID | Name | Definition |
|---|---|---|
| JSI | Judge Severity Index | $\text{JSI}(j) = \mu_j - \mu_{\text{grand}}$ |
| SIT | Score Inflation Trend | OLS slope $\hat{\beta}_1$ of $s_{j,t} = \beta_0 + \beta_1 t + \varepsilon_t$ |
| IMAM | Inter-Model Agreement | Pearson $\rho(\bar{s}_{i \to \cdot}, \bar{s}_{j \to \cdot})$ per judge pair |
| DARS | Drift in A-Received Scores | Rolling $\mu$ and OLS slope of scores where `target_id="A"` |
| DAGS | Drift in A-Given Scores | Rolling $\mu$ and OLS slope of scores where `judge_id="A"` |
| RI | Reciprocity Index | $\rho(s_{A \to X,t},\, s_{X \to A,t})$ per peer $X$ |
| VD | Variance Decay | Rolling $\sigma^2$ of all scores over time |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from src.metrics import compute_all_metrics, save_metrics_summary, MetricsSummary

scores_log_path = REPO_ROOT / "logs" / "scores_log.jsonl"

if scores_log_path.exists() and scores_log_path.stat().st_size > 0:
    # ── Load scores into DataFrame ────────────────────────────────────────
    df = pd.read_json(scores_log_path, lines=True)
    print(f"Score records loaded: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Judge IDs:  {sorted(df['judge_id'].unique())}")
    print(f"Target IDs: {sorted(df['target_id'].unique())}")
    print(f"Iteration range: {df['iteration_id'].min()} – {df['iteration_id'].max()}")

    # ── Compute all 7 metrics ─────────────────────────────────────────────
    summary: MetricsSummary = compute_all_metrics(scores_log_path)
    results_dir = REPO_ROOT / "results"
    results_dir.mkdir(exist_ok=True)
    save_metrics_summary(summary, results_dir)

    # ── Display as table ──────────────────────────────────────────────────
    print("\n--- Judge Severity Index (JSI) ---")
    for judge, jsi in summary.jsi.items():
        direction = "↑ lenient" if jsi > 0 else "↓ strict"
        print(f"  Judge {judge}: JSI = {jsi:+.3f}  {direction}")

    print("\n--- Score Inflation Trend (SIT, OLS slope per judge) ---")
    for judge, sit in summary.sit.items():
        trend = "inflating" if sit > 0 else "deflating"
        print(f"  Judge {judge}: β₁ = {sit:+.5f}  ({trend})")

    print("\n--- DARS (Drift in Model A's received scores) ---")
    print(f"  Slope:  {summary.dars_slope:+.5f}")
    print(f"  Δ score (first→last quartile): {summary.dars_quartile_delta:+.3f}")

    print("\n--- DAGS (Drift in Model A's given scores) ---")
    print(f"  Slope:  {summary.dags_slope:+.5f}")

    print("\n--- Reciprocity Index (RI) per peer ---")
    for peer, ri in summary.reciprocity.items():
        print(f"  A ↔ {peer}: ρ = {ri:+.4f}")

else:
    print("No scores_log.jsonl found — run the experiment first.")
    print("Generating synthetic data for metric demonstration...")

    # ── Synthetic data demo ────────────────────────────────────────────────
    rng = np.random.default_rng(42)
    n_iters = 100
    records = []
    for it in range(n_iters):
        # Peers judge A (slight inflation over time)
        for pos, judge in enumerate(["B", "C", "D"]):
            drift = 0.05 * it       # synthetic drift
            score = float(np.clip(rng.normal(65 + drift, 8), 0, 100))
            records.append({
                "iteration_id": it, "judge_id": judge, "target_id": "A",
                "weighted_total": score,
                "dimensions": {d: score for d in RUBRIC_DIMENSIONS},
                "parse_success": True, "asymmetry_position": pos,
            })
        # A judges peers (slight deflation over time)
        for pos, target in enumerate(["B", "C", "D"]):
            drift = -0.03 * it
            score = float(np.clip(rng.normal(60 + drift, 9), 0, 100))
            records.append({
                "iteration_id": it, "judge_id": "A", "target_id": target,
                "weighted_total": score,
                "dimensions": {d: score for d in RUBRIC_DIMENSIONS},
                "parse_success": True, "asymmetry_position": 3 + pos,
            })

    synthetic_df = pd.DataFrame(records)
    print(f"Synthetic records: {len(synthetic_df)}")
    print(f"Mean score (A as target): "
          f"{synthetic_df[synthetic_df.target_id=='A'].weighted_total.mean():.2f}")
    print(f"Mean score (A as judge):  "
          f"{synthetic_df[synthetic_df.judge_id=='A'].weighted_total.mean():.2f}")

## Section 14 — Visualization Suite

Eight publication-grade figures (300 DPI, PNG + PDF, `DejaVu Serif` font):

| Figure | File | Content |
|---|---|---|
| 01 | `01_score_evolution.png` | Rolling mean of all scores over iterations |
| 02 | `02_drift_curve_model_a.png` | A's received vs given scores, side-by-side |
| 03 | `03_judge_severity_index.png` | JSI bar chart per judge |
| 04 | `04_scoring_matrix_heatmap.png` | Mean score heatmap: judge × target |
| 05 | `05_inflation_trend.png` | Per-judge OLS trend lines over time |
| 06 | `06_variance_over_time.png` | Rolling σ² of all scores |
| 07 | `07_inter_model_agreement.png` | Pearson ρ matrix between judges |
| 08 | `08_reciprocity_scatter.png` | Scatter: A→X score vs X→A score per peer |

In [ ]:
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for headless render
import matplotlib.pyplot as plt
import matplotlib.pyplot as _plt
from src.visualization import render_all_figures
from src.metrics import compute_all_metrics, MetricsSummary

figures_dir = REPO_ROOT / "figures"
figures_dir.mkdir(exist_ok=True)

if scores_log_path.exists() and scores_log_path.stat().st_size > 0:
    # ── Load metrics summary ──────────────────────────────────────────────
    summary = compute_all_metrics(scores_log_path)

    # ── Render all 8 figures ──────────────────────────────────────────────
    render_all_figures(
        scores_log_path=scores_log_path,
        metrics_summary=summary,
        figures_dir=figures_dir,
    )

    # ── Display inline ────────────────────────────────────────────────────
    saved_figures = sorted(figures_dir.glob("*.png"))
    print(f"Figures saved to: {figures_dir}")
    print(f"Count: {len(saved_figures)}")
    for fig_path in saved_figures:
        print(f"  {fig_path.name}")
        from IPython.display import Image, display
        display(Image(filename=str(fig_path), width=700))

else:
    # ── Synthetic visualisation demo (pre-run) ───────────────────────────
    print("No experiment data found. Rendering demo figures with synthetic data.")

    rng2  = np.random.default_rng(42)
    iters = np.arange(100)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Panel 1: rolling mean scores per judge
    ax = axes[0]
    for judge, color in zip(["B", "C", "D", "A"], ["#66c2a5","#fc8d62","#8da0cb","#e78ac3"]):
        drift = 0.04 if judge != "A" else -0.03
        raw   = rng2.normal(65 + drift * iters, 8)
        roll  = pd.Series(raw).rolling(10, min_periods=1).mean()
        ax.plot(iters, roll, label=f"Scores given to A by {judge}" if judge != "A"
                else "Scores given by A", color=color, lw=1.8)
    ax.set_title("Score Evolution (synthetic)", fontsize=11)
    ax.set_xlabel("Iteration"); ax.set_ylabel("Rolling mean score")
    ax.legend(fontsize=8); ax.set_ylim(40, 90)

    # Panel 2: JSI bar chart
    ax = axes[1]
    jsi_vals = {"B": 3.2, "C": 1.1, "D": -0.8, "A": -2.5}
    colors   = ["#66c2a5" if v >= 0 else "#fc8d62" for v in jsi_vals.values()]
    bars     = ax.bar(jsi_vals.keys(), jsi_vals.values(), color=colors, edgecolor="k", lw=0.7)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title("Judge Severity Index (synthetic)", fontsize=11)
    ax.set_xlabel("Judge"); ax.set_ylabel("JSI = μ_j − μ_grand")

    # Panel 3: reciprocity scatter
    ax = axes[2]
    for peer, mrk in zip(["B", "C", "D"], ["o", "s", "^"]):
        x = rng2.normal(62, 10, 100)
        y = 0.4 * x + rng2.normal(0, 8, 100)
        ax.scatter(x, y, alpha=0.5, s=20, marker=mrk, label=f"A↔{peer}")
    ax.set_title("Reciprocity Index (synthetic)", fontsize=11)
    ax.set_xlabel("Score A→peer"); ax.set_ylabel("Score peer→A")
    ax.legend(fontsize=8)

    plt.tight_layout()
    demo_fig_path = figures_dir / "demo_synthetic_preview.png"
    plt.savefig(demo_fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Demo figure saved → {demo_fig_path}")

## Section 15 — Dry Run Mode & Consistency Assertions

The dry-run mode replaces real model inference with fast mock generators, enabling full pipeline validation in under 60 seconds on any machine.  
12 assertions verify end-to-end invariants before any GPU compute is committed.

In [ ]:
import random as _random_mod
from tasks.rubrics import parse_judge_output
from tasks.templates import TaskRegistry

def mock_generate(prompt: str, seed: int, max_new_tokens: int = 256) -> str:
    """Deterministic mock generator — no GPU required."""
    rng_local = _random_mod.Random(seed)
    words = [
        "policy", "analysis", "context", "framework", "decision",
        "impact", "consideration", "approach", "evidence", "outcome",
    ]
    n = rng_local.randint(40, 80)
    return " ".join(rng_local.choices(words, k=n)) + "."

def mock_judge(task_prompt: str, response_text: str,
               rubric_dims: dict, seed: int) -> dict:
    """Deterministic mock judge — returns plausible JSON scores."""
    rng_local = _random_mod.Random(seed)
    scores = {
        dim: round(rng_local.gauss(65, 10))
        for dim in rubric_dims
    }
    # Clamp to [0, 100]
    scores = {k: max(0, min(100, v)) for k, v in scores.items()}
    weighted = sum(scores[d] * w for d, w in rubric_dims.items())
    raw_output = (
        "```json\n"
        + json.dumps(scores, indent=2)
        + "\n```\n"
        + f"Weighted total: {weighted:.1f}"
    )
    return parse_judge_output(raw_output, rubric_dims, attempt=0)

# ── Run 12 consistency assertions ────────────────────────────────────────
errors: list[str] = []
N_DRY  = 5   # small number for fast validation

dry_reg  = TaskRegistry(global_seed=GLOBAL_SEED)
dry_vars = list(dry_reg.iter_all(num_templates=NUM_TEMPLATES, num_variations=N_DRY))

seen_iter_ids: set[int] = set()
iter_seeds_seen: set[int] = set()

for v in dry_vars:
    # Assertion 1: iteration_id unique
    if v.iteration_id in seen_iter_ids:
        errors.append(f"Duplicate iteration_id {v.iteration_id}")
    seen_iter_ids.add(v.iteration_id)

    # Assertion 2: template_id in expected range
    if not (0 <= v.template_id < NUM_TEMPLATES):
        errors.append(f"template_id {v.template_id} out of range")

    # Assertion 3: variation_id in expected range
    if not (0 <= v.variation_id < N_DRY):
        errors.append(f"variation_id {v.variation_id} out of range {N_DRY}")

    # Assertion 4: prompt is non-empty string
    if not isinstance(v.prompt, str) or len(v.prompt) < 20:
        errors.append(f"Malformed prompt at iter {v.iteration_id}")

    # Run mock generation for each model
    for m_idx, model_id in enumerate(MODEL_IDS):
        iter_seed = derive_iteration_seed(GLOBAL_SEED, v.iteration_id)
        m_seed    = derive_model_seed(iter_seed, m_idx)
        response  = mock_generate(v.prompt, m_seed)

        # Assertion 5: response non-empty
        if not response:
            errors.append(f"Empty response: model={model_id} iter={v.iteration_id}")

    # Run mock judging for peer → A direction
    for pos, judge_id in enumerate(PEER_IDS):
        j_seed = derive_model_seed(derive_iteration_seed(GLOBAL_SEED, v.iteration_id), pos + 4)
        jscore = mock_judge(v.prompt, "sample response text", RUBRIC_DIMENSIONS, j_seed)

        # Assertion 6: weighted_total in [0, 100]
        if not (0.0 <= jscore.weighted_total <= 100.0):
            errors.append(f"Score out of bounds: {jscore.weighted_total}")

        # Assertion 7: all rubric dimensions present in output
        if set(jscore.dimensions.keys()) != set(RUBRIC_DIMENSIONS.keys()):
            errors.append(f"Missing dimensions at iter {v.iteration_id}")

# Assertion 8: total iterations = templates × variations
expected_total = NUM_TEMPLATES * N_DRY
if len(seen_iter_ids) != expected_total:
    errors.append(f"Expected {expected_total} iterations, got {len(seen_iter_ids)}")

# Assertion 9: seed derivation is deterministic
seed_a = derive_iteration_seed(GLOBAL_SEED, 42)
seed_b = derive_iteration_seed(GLOBAL_SEED, 42)
if seed_a != seed_b:
    errors.append("Seed derivation is non-deterministic")

# Assertion 10: global seed = 42
if GLOBAL_SEED != 42:
    errors.append(f"GLOBAL_SEED must be 42, got {GLOBAL_SEED}")

# Assertion 11: rubric weights sum to 1
weight_sum = sum(RUBRIC_DIMENSIONS.values())
if abs(weight_sum - 1.0) > 1e-9:
    errors.append(f"Rubric weights sum to {weight_sum}, expected 1.0")

# Assertion 12: model list has exactly 4 entries
if len(MODEL_IDS) != 4:
    errors.append(f"Expected 4 models, got {len(MODEL_IDS)}")

# ── Report ────────────────────────────────────────────────────────────────
if errors:
    print(f"FAILED — {len(errors)} assertion(s):")
    for e in errors:
        print(f"  ✗ {e}")
else:
    print(f"All 12 consistency assertions passed ✓")
    print(f"  Iterations validated : {len(seen_iter_ids)}")
    print(f"  Models per iteration : {len(MODEL_IDS)}")
    print(f"  Mock judgements run  : {len(seen_iter_ids) * len(PEER_IDS) * 2}")

## Section 16 — Main Experiment Loop with Resume Support

The main loop iterates over all 1000 `(template_id, variation_id)` pairs in deterministic order.  
On each iteration: generate → judge (asymmetry protocol) → log → checkpoint.  
Set `RESUME = True` to pick up from the last completed checkpoint.

In [ ]:
"""
Main experiment loop.

Set DRY_RUN = True  to use mock generators (no GPU required, completes in ~minutes).
Set DRY_RUN = False to run full experiment (requires T4 / 16 GB VRAM).
Set RESUME  = True  to continue from last checkpoint.
"""

import time, json

DRY_RUN      : bool = True   # ← change to False for full run
RESUME       : bool = False
N_ITER_LIMIT : int  = 10 if DRY_RUN else 1000   # cap iterations in dry mode

# ── Seed everything ────────────────────────────────────────────────────
def set_global_seed(seed: int) -> None:
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(GLOBAL_SEED)

# ── Load checkpoint ────────────────────────────────────────────────────
ckpt_dir2 = REPO_ROOT / "checkpoints"
ckpt_dir2.mkdir(exist_ok=True)
cm2  = CheckpointManager(checkpoint_dir=ckpt_dir2, total_iterations=N_ITER_LIMIT)
ckpt3 = cm2.load_or_create()

start_iter = ckpt3.next_iteration if RESUME else 0
print(f"DRY_RUN={DRY_RUN} | RESUME={RESUME}")
print(f"Starting at iteration {start_iter}/{N_ITER_LIMIT}")

# ── Logger ─────────────────────────────────────────────────────────────
run_id     = f"{'dry_run' if DRY_RUN else 'full_run'}_{int(time.time())}"
exp_logger = ExperimentLogger(log_dir=REPO_ROOT / "logs", run_id=run_id)
exp_logger.log_experiment_start(
    config={"dry_run": DRY_RUN, "n_iterations": N_ITER_LIMIT,
            "global_seed": GLOBAL_SEED, "models": MODEL_IDS}
)

# ── Main loop ──────────────────────────────────────────────────────────
completed = 0
for var in dry_reg.iter_all(num_templates=NUM_TEMPLATES,
                             num_variations=N_DRY if DRY_RUN else NUM_VARIATIONS):
    if var.iteration_id < start_iter:
        continue
    if var.iteration_id >= N_ITER_LIMIT:
        break

    iter_seed = derive_iteration_seed(GLOBAL_SEED, var.iteration_id)
    t0        = time.time()

    # ── Phase 1: Generate responses ────────────────────────────────────
    responses: dict[str, str] = {}
    for m_idx, model_id in enumerate(MODEL_IDS):
        m_seed = derive_model_seed(iter_seed, m_idx)
        if DRY_RUN:
            resp = mock_generate(var.prompt, m_seed)
        else:
            # Replace with: resp = model_handles[model_id].generate(...)
            raise RuntimeError("Set DRY_RUN=True or connect model handles.")
        responses[model_id] = resp
        exp_logger.log_response(
            iteration_id=var.iteration_id,
            model_id=model_id,
            response_text=resp,
            generation_seed=m_seed,
            generation_time_sec=0.0,
        )

    # ── Phase 2a: Peers judge A ────────────────────────────────────────
    for pos, judge_id in enumerate(PEER_IDS):
        j_seed  = derive_model_seed(iter_seed, len(MODEL_IDS) + pos)
        jscore  = mock_judge(var.prompt, responses["A"], RUBRIC_DIMENSIONS, j_seed)
        exp_logger.log_score(
            iteration_id=var.iteration_id,
            judge_id=judge_id,
            target_id="A",
            weighted_total=jscore.weighted_total,
            dimensions=jscore.dimensions,
            parse_success=jscore.parse_success,
            asymmetry_position=pos,
        )

    # ── Phase 2b: A judges peers ────────────────────────────────────────
    for pos, target_id in enumerate(PEER_IDS):
        j_seed  = derive_model_seed(iter_seed, len(MODEL_IDS) + len(PEER_IDS) + pos)
        jscore  = mock_judge(var.prompt, responses[target_id], RUBRIC_DIMENSIONS, j_seed)
        exp_logger.log_score(
            iteration_id=var.iteration_id,
            judge_id="A",
            target_id=target_id,
            weighted_total=jscore.weighted_total,
            dimensions=jscore.dimensions,
            parse_success=jscore.parse_success,
            asymmetry_position=len(PEER_IDS) + pos,
        )

    # ── Checkpoint ─────────────────────────────────────────────────────
    elapsed   = time.time() - t0
    ckpt3     = cm2.update(ckpt3, completed_iteration=var.iteration_id,
                           duration_sec=elapsed)
    cm2.save(ckpt3)
    completed += 1

exp_logger.log_experiment_end(
    status="complete",
    iterations_run=completed,
    total_wall_sec=sum(ckpt3.iteration_durations_sec.values()),
)

print(f"\nLoop complete. Iterations processed: {completed}")
print(f"Average time/iter: "
      f"{sum(ckpt3.iteration_durations_sec.values())/max(completed,1)*1000:.1f} ms")
print(f"Logs written to: {REPO_ROOT / 'logs'}")

# ── Save completion marker ─────────────────────────────────────────────
(REPO_ROOT / "results").mkdir(exist_ok=True)
completion_record = {
    "run_id":           run_id,
    "dry_run":          DRY_RUN,
    "iterations":       completed,
    "global_seed":      GLOBAL_SEED,
    "completed_at":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(REPO_ROOT / "results" / "experiment_complete.json").write_text(
    json.dumps(completion_record, indent=2)
)
print(f"\nexperiment_complete.json written → {REPO_ROOT / 'results' / 'experiment_complete.json'}")